# 03 · Data cleaning

Inspects the raw tables for the issues that matter for downstream
analysis, documents the decision made for each, and writes clean,
analysis-ready tables to `data/processed/`.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("..").resolve()
DB_PATH = ROOT / "db" / "olist.db"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
orders = pd.read_sql_query("SELECT * FROM orders", conn)
customers = pd.read_sql_query("SELECT * FROM customers", conn)
items = pd.read_sql_query("SELECT * FROM order_items", conn)
payments = pd.read_sql_query("SELECT * FROM order_payments", conn)
reviews = pd.read_sql_query("SELECT * FROM order_reviews", conn)
products = pd.read_sql_query("SELECT * FROM products", conn)
sellers = pd.read_sql_query("SELECT * FROM sellers", conn)
cat_trans = pd.read_sql_query("SELECT * FROM category_translation", conn)
geo = pd.read_sql_query("SELECT * FROM geolocation", conn)

## 1. Parse dates + check order status / missing delivery timestamps

In [2]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

print(f"Order date range: {orders['order_purchase_timestamp'].min().date()} "
      f"to {orders['order_purchase_timestamp'].max().date()}")

Order date range: 2016-09-04 to 2018-10-17


In [3]:
null_delivered = orders["order_delivered_customer_date"].isna().sum()
status_of_null_delivered = (
    orders.loc[orders["order_delivered_customer_date"].isna(), "order_status"]
    .value_counts()
)
print(f"Missing order_delivered_customer_date: {null_delivered:,} rows")
print(status_of_null_delivered)
# Decision: these are undelivered orders (canceled/unavailable/etc.), not
# data errors. All delivery-time analysis filters to order_status ==
# 'delivered' AND order_delivered_customer_date not null.

Missing order_delivered_customer_date: 2,965 rows
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [4]:
status_counts = orders["order_status"].value_counts()
print("Order status breakdown:")
print((status_counts / len(orders) * 100).round(2))

Order status breakdown:
order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: count, dtype: float64


## 2. `geolocation` duplicates
Raw file has many exact duplicate rows and multiple lat/lng samples per
zip prefix, so a raw join would fan out order rows. Decision: aggregate
to one row per state (mean lat/lng) for map visuals.

In [5]:
geo_dupes = geo.duplicated().sum()
print(f"geolocation: {len(geo):,} rows, {geo_dupes:,} exact duplicates "
      f"({geo_dupes/len(geo)*100:.1f}%)")

geo_state = (
    geo.groupby("geolocation_state")[["geolocation_lat", "geolocation_lng"]]
    .mean()
    .reset_index()
    .rename(columns={"geolocation_state": "state",
                      "geolocation_lat": "lat", "geolocation_lng": "lng"})
)
geo_state.to_csv(PROCESSED / "geolocation_state_centroids.csv", index=False)

geolocation: 1,000,163 rows, 261,831 exact duplicates (26.2%)


## 3. Category translation gaps
A couple of category names in `products` have no row in the English
translation table. Mapped by hand rather than dropped.

In [6]:
prod_cats = set(products["product_category_name"].dropna().unique())
trans_cats = set(cat_trans["product_category_name"].unique())
missing_cats = prod_cats - trans_cats

manual_map = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "kitchen_portable_appliances",
}
for cat in sorted(missing_cats):
    n = (products["product_category_name"] == cat).sum()
    print(f"  '{cat}' ({n} products) -> mapped to '{manual_map.get(cat, cat)}'")

extra_rows = pd.DataFrame(
    [{"product_category_name": k, "product_category_name_english": v}
     for k, v in manual_map.items()]
)
cat_trans_fixed = pd.concat([cat_trans, extra_rows], ignore_index=True)
cat_trans_fixed.to_csv(PROCESSED / "category_translation_fixed.csv", index=False)

products_fixed = products.copy()
products_fixed["product_category_name"] = products_fixed["product_category_name"].fillna("unknown")
products_fixed = products_fixed.merge(cat_trans_fixed, on="product_category_name", how="left")
products_fixed["product_category_name_english"] = (
    products_fixed["product_category_name_english"].fillna("unknown")
)

  'pc_gamer' (3 products) -> mapped to 'pc_gamer'
  'portateis_cozinha_e_preparadores_de_alimentos' (10 products) -> mapped to 'kitchen_portable_appliances'


## 4. `customer_id` vs `customer_unique_id`
Olist re-issues a new `customer_id` per order — the same shopper gets a
new ID every time they buy. `customer_unique_id` is the real person.
**All RFM / cohort / retention analysis keys off `customer_unique_id`.**

In [7]:
n_customer_id = customers["customer_id"].nunique()
n_unique_person = customers["customer_unique_id"].nunique()
repeat_people = (customers["customer_unique_id"].value_counts() > 1).sum()
print(f"{n_customer_id:,} customer_id values vs {n_unique_person:,} unique people")
print(f"Repeat buyers: {repeat_people:,} ({repeat_people/n_unique_person*100:.2f}%)")

99,441 customer_id values vs 96,096 unique people
Repeat buyers: 2,997 (3.12%)


## 5. Reviews: keep the latest answer per order

In [8]:
reviews_sorted = reviews.copy()
reviews_sorted["review_answer_timestamp"] = pd.to_datetime(
    reviews_sorted["review_answer_timestamp"], errors="coerce"
)
reviews_dedup = (
    reviews_sorted.sort_values("review_answer_timestamp")
    .drop_duplicates(subset="order_id", keep="last")
    [["order_id", "review_score", "review_creation_date", "review_answer_timestamp"]]
)
dupe_review_orders = (reviews["order_id"].value_counts() > 1).sum()
print(f"{dupe_review_orders:,} orders had more than one review row (kept latest only)")

555 orders had more than one review row (kept latest only)


## Build clean fact tables

In [9]:
order_value = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    merchandise_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
).reset_index()
order_value["order_total_value"] = order_value["merchandise_value"] + order_value["freight_value"]

orders_customers = orders.merge(
    customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
    on="customer_id", how="left",
)

orders_clean = (
    orders_customers
    .merge(order_value, on="order_id", how="left")
    .merge(reviews_dedup, on="order_id", how="left")
)

orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400
orders_clean["estimate_days"] = (
    orders_clean["order_estimated_delivery_date"] - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400
# positive = delivered earlier than estimated; negative = late
orders_clean["days_early_vs_estimate"] = (
    orders_clean["order_estimated_delivery_date"] - orders_clean["order_delivered_customer_date"]
).dt.total_seconds() / 86400
orders_clean["is_late"] = orders_clean["days_early_vs_estimate"] < 0

orders_clean.to_parquet(PROCESSED / "orders_clean.parquet", index=False)

items_clean = items.merge(
    products_fixed[["product_id", "product_category_name_english"]],
    on="product_id", how="left",
).merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left",
)
items_clean.to_parquet(PROCESSED / "order_items_clean.parquet", index=False)

conn.close()
print("Clean outputs written to data/processed/")
orders_clean.head()

Clean outputs written to data/processed/


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,...,merchandise_value,freight_value,order_total_value,review_score,review_creation_date,review_answer_timestamp,delivery_days,estimate_days,days_early_vs_estimate,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,...,29.99,8.72,38.71,4,2017-10-11 00:00:00,2017-10-12 03:43:48,8.436574,15.544063,7.107488,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,...,118.70,22.76,141.46,4,2018-08-08 00:00:00,2018-08-08 18:37:50,13.782037,19.137766,5.355729,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,...,159.90,19.22,179.12,5,2018-08-18 00:00:00,2018-08-22 19:07:58,9.394213,26.639711,17.245498,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,...,45.00,27.20,72.20,5,2017-12-03 00:00:00,2017-12-05 19:21:58,13.208750,26.188819,12.980069,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,...,19.90,8.72,28.62,5,2018-02-17 00:00:00,2018-02-18 13:02:51,2.873877,12.112049,9.238171,False
